In [1]:
import pyterrier as pt
import matplotlib.pyplot as plt
import pandas as pd
import os
from ir_measures import *

run_dir = "./runs/"

In [2]:
docs = pd.read_pickle("/nfs/primary/sas_reranker/ohsumed_docs_w_t5base_sensitivity.pkl")
queries = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/queries.pkl")
qrels = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/qrels.pkl")

In [12]:
run_files = [pt.io.read_results(f"{run_dir}/{f}") for f in os.listdir(run_dir) if not f.endswith("checkpoints")]
run_names = [f for f in os.listdir(run_dir) if not f.endswith("checkpoints")]

In [13]:
run_names

['qenc_tilde_baseline_denc_tilde_strategy1_group7',
 'qenc_tilde_baseline_denc_tilde_strategy1_group3',
 'qenc_tilde_baseline_denc_tilde_strategy1_group2',
 'qenc_tilde_baseline_denc_tilde_strategy3_group2',
 'qenc_tilde_baseline_denc_tilde_strategy2_group2',
 'qenc_tilde_baseline_denc_tilde_strategy2_group4',
 'qenc_tilde_baseline_denc_tilde_strategy3_group3',
 'qenc_tilde_baseline_denc_tilde_strategy3_group5',
 'qenc_tilde_baseline_denc_tilde_strategy3_group4',
 'qenc_tilde_baseline_denc_tilde_strategy2_group7',
 'qenc_tilde_baseline_denc_tilde_strategy2_group8',
 'qenc_tilde_baseline_denc_tilde_strategy1_group8',
 'qenc_tilde_baseline_denc_tilde_strategy3_group6',
 'qenc_tilde_baseline_denc_tilde_strategy3_group8',
 'qenc_tilde_baseline_denc_tilde_strategy1_group5',
 'qenc_tilde_baseline_denc_tilde_strategy2_group5',
 'qenc_tilde_baseline_denc_tilde_strategy3_group7',
 'qenc_tilde_baseline_denc_tilde_strategy2_group6',
 'qenc_tilde_baseline_denc_tilde_strategy1_group4',
 'qenc_tilde

In [14]:
def _sens_docs(qrels, run):
    if "sensitivity" in run.columns:
        run = run.drop(columns = ["sensitivity"])
    merged = pd.merge(run, docs, left_on = "doc_id", right_on = "docno")
    return merged.sensitivity.sum()
    
import ir_measures
sens_docs = ir_measures.define_byquery(
    _sens_docs, 
    name="sens_docs")

In [15]:
retrieval_results = pt.Experiment(
    run_files,
    queries,
    qrels,
    eval_metrics=[nDCG@10, sens_docs@10],
    names = run_names
)

In [16]:
retrieval_results

,name,nDCG@10,sens_docs@10
0,qenc_tilde_baseline_denc_tilde_strategy1_group7,0.412869,1.264151
1,qenc_tilde_baseline_denc_tilde_strategy1_group3,0.412869,1.264151
2,qenc_tilde_baseline_denc_tilde_strategy1_group2,0.412869,1.264151
3,qenc_tilde_baseline_denc_tilde_strategy3_group2,0.412869,1.264151
4,qenc_tilde_baseline_denc_tilde_strategy2_group2,0.412869,1.264151
5,qenc_tilde_baseline_denc_tilde_strategy2_group4,0.412869,1.264151
6,qenc_tilde_baseline_denc_tilde_strategy3_group3,0.412869,1.264151
7,qenc_tilde_baseline_denc_tilde_strategy3_group5,0.412704,1.264151
8,qenc_tilde_baseline_denc_tilde_strategy3_group4,0.412704,1.264151
9,qenc_tilde_baseline_denc_tilde_strategy2_group7,0.412704,1.264151


In [6]:
# Extract model prefixes and group sizes from 'name'
retrieval_results['model'] = retrieval_results['name'].str.extract(r"^(splade_max|tilde|unicoil)")
retrieval_results['group'] = retrieval_results['name'].str.extract(r"(group\d+)")
retrieval_results = retrieval_results.dropna()
retrieval_results['strategy'] = retrieval_results['name'].str.extract(r"(strategy\d+)")

In [7]:
retrieval_results

,name,model,group,strategy
